In [1]:
import models
import numpy as np
import matplotlib.pyplot as plt
import datavis as dv
from scipy.ndimage import gaussian_laplace as gl
from scipy.ndimage import gaussian_filter
from scipy.ndimage import gaussian_filter1d
import firstclassifier as fc

In [2]:
# generate a dataset
T = 1000
datasize = 20
trials = 400
bw=20

# generate a random set of ramp parameters
beta = np.random.uniform(0, 4, datasize)
ln_sig = np.random.uniform(np.log(0.04), np.log(4), datasize)
ramp_params = np.column_stack((beta, np.exp(ln_sig)))

# generate a random set of step parameters
r = np.random.uniform(0.5, 6, datasize)
m = np.random.uniform(T/4, 3*T/4, datasize)
step_params = np.column_stack((m, r))

x_init = np.random.uniform(0, 0.5, datasize*2)


# generate simulations of each parameter set
ramp_dataset = np.array([models.RampModel(beta=ramp_params[i,0],
                                      sigma=ramp_params[i,1],
                                      x0 = x_init[i]).simulate(Ntrials=trials,
                                                               T=T)[0]
                                      for i in range(datasize)
                                      ]
                                    )
                                    
step_dataset = np.array([models.StepModel(m=step_params[j,0],
                                      r=step_params[j,1],
                                      x0 = x_init[j+datasize]).simulate(Ntrials=trials,
                                                                   T=T)[0]
                                      for j in range(datasize)
                                      ]
)

dataset = np.vstack((ramp_dataset, step_dataset))

In [3]:
# classify based on maximum fano factor
mff_classifier = fc.genFanoClassifyMax(dataset, datasize)
print(mff_classifier)
print(fc.accuracy(mff_classifier, datasize))

[0 1 0 0 0 0 1 1 0 0 0 0 0 1 1 0 0 0 0 0 1 1 1 0 0 0 1 1 0 1 1 1 1 0 1 1 1
 1 1 1]
(np.float64(0.75), array([ True, False,  True,  True,  True,  True, False, False,  True,
        True,  True,  True,  True, False, False,  True,  True,  True,
        True,  True,  True,  True,  True, False, False, False,  True,
        True, False,  True,  True,  True,  True, False,  True,  True,
        True,  True,  True,  True]))


In [4]:
# classify based on maximum derivative of fano factor
max_gs, mfd = fc.maxFanoDeriv(dataset, datasize, threshold=0.0005)
print(mfd)
print(fc.accuracy(mfd, datasize))

[0 0 0 1 0 0 1 0 1 0 0 0 0 1 1 0 0 0 0 1 1 1 1 0 1 1 1 1 0 1 1 1 1 0 1 1 1
 1 1 1]
(np.float64(0.775), array([ True,  True,  True, False,  True,  True, False,  True, False,
        True,  True,  True,  True, False, False,  True,  True,  True,
        True, False,  True,  True,  True, False,  True,  True,  True,
        True, False,  True,  True,  True,  True, False,  True,  True,
        True,  True,  True,  True]))
